In [1]:
import os
import torch

In [12]:
with open("names.txt", "r+") as file:
    contents = file.read()

split_lines = contents.splitlines()
char_set = sorted(set(''.join(split_lines)))


In [102]:
contexts = ['a', 'n', 'n']
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), requires_grad=True, generator=g)

block_size = 3

ch2i = {}
i2ch = {}
ch2i['.'] = 0
i2ch[0] = '.'

for i, v in enumerate(char_set):
    ch2i[v] = i + 1
    i2ch[i + 1] = v

combine = []
for v in contexts:
    index = ch2i[v]
    lines = C[index]
    combine += lines

converted = []
X = []
Y = []
for v in split_lines:
    added = list(v)
    context = ['.'] * block_size
    for target in added + ['.']:
        nums = []
        for c in context:
            nums.append(ch2i[c])
        X.append(nums)
        Y.append(ch2i[target])
        converted.append((context, target))
        context = context[1:] + [target]
        
tx = torch.tensor(X)
ty = torch.tensor(Y)

# print(converted[:10])
# print(tx)
# print(ty)
# print(tx.shape)

# print(len(split_lines))
# total = 0
# for i, v in enumerate(split_lines):
#     total += len(v)
# print( total + len(split_lines))
# print(C[5].shape)
# print(C[[5, 13, 13]], C[[5, 13, 13]].shape)

torch.cat([C[5], C[13], C[13]])

dim_N_6 = []
# for x in tx[3:4]:
#     tmp = []
#     for i in x:
#         tmp.append(C[i])
#     conc = torch.cat(tmp)
#     dim_N_6.append(conc)

emb = C[tx]
dim_N_6 = torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1)

#最终写法，当前最优解
# dim_N_6 = C[tx].view(-1, 6)

# print(dim_N_6[:3])

W1 = torch.randn((6, 100), requires_grad=True)
B1 = torch.randn((100,), requires_grad=True)

hidden = torch.tanh(dim_N_6 @ W1 + B1)
hidden

print(hidden)

W2 = torch.randn((100, 27), requires_grad=True)
B2 = torch.randn((27,), requires_grad=True)

logits = hidden @ W2 + B2
logits

F = torch.nn.functional
loss = F.cross_entropy(logits, ty)
loss

a = torch.tensor([-100, -3, 0, 100]).float()
b = torch.tensor(2)
result = a.exp() / a.exp().sum(0, keepdim=True)
print(result)
print(F.cross_entropy(a, b))

tensor([[ 0.9972,  0.9951,  0.9996,  ...,  0.4389, -0.9900,  0.9898],
        [-0.9551,  0.9989,  0.8318,  ..., -0.7662, -0.9993,  0.9688],
        [ 1.0000,  0.9540,  0.9949,  ..., -0.9917, -0.9175,  0.9163],
        ...,
        [-0.9889, -0.6359, -0.9659,  ..., -0.9718, -0.9922,  0.0834],
        [ 0.8584,  0.6199, -0.6605,  ..., -1.0000,  0.7638, -0.9600],
        [-0.9991, -0.9979, -0.9963,  ..., -0.9766, -0.9219,  0.0304]],
       grad_fn=<TanhBackward0>)
tensor([0., 0., 0., nan])
tensor(100.)
